# Create DVG ranks

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.preprocessing import MinMaxScaler

from utils.plotting import granger_color

In [ ]:
interpolated_ts_df = pd.read_csv('data/outputs/plus1_log10_linear_imputation/lin_interpolated_ts_df.csv', index_col=0)

In [ ]:
n_timepoints = len(interpolated_ts_df)
n_splits = 8
fold_size = 3

fold_max_dpi = {}

ts_data = interpolated_ts_df.fillna(0)

test_ts_data = {}
train_ts_data = {}

for i in range(n_splits):
    start = n_timepoints - (i + 1) * fold_size
    end = n_timepoints - i * fold_size
    test_ts_data[n_splits - i] = ts_data.iloc[start:end]
    train_ts_data[n_splits - i] = ts_data.iloc[:start]
    print(f"Fold {n_splits - i}: {test_ts_data[n_splits - i].index.tolist()}")
    fold_max_dpi[n_splits - i] = test_ts_data[n_splits - i].index.max()
    
fold_max_dpi


In [ ]:
output_prefix = 'data/outputs/plus1_log10_linear_imputation'
forecast_output_prefix = f'{output_prefix}_{n_splits}_folds_size{fold_size}'

In [ ]:
summarized_fold_predictions_bh = pickle.load(open(f"{forecast_output_prefix}/summarized_fold_predictions_bh.pkl", "rb"))
summarized_fold_maes_bh = pickle.load(open(f"{forecast_output_prefix}/summarized_fold_maes_bh.pkl", "rb"))
avg_summarized_maes_corrected = pickle.load(open(f"{forecast_output_prefix}/avg_summarized_maes_corrected.pkl", "rb"))


In [ ]:
bh_summary_df = {}
gc_prediction_summary_corrected = {}


for lag in range(1, 2):
    bh_summary_df[lag] = pd.read_csv(f'{output_prefix}/gc_summary_df_corrected_lag{lag}.csv', index_col=0)
    gc_prediction_summary_corrected[lag] = pd.read_csv(f'{output_prefix}/gc_prediction_summary_corrected_lag{lag}.csv')

In [ ]:
def plot_dvg_forecast_and_mae(dvg_id, lag=1, figsize=(8,4)):
    preds_df = summarized_fold_predictions_bh[lag]
    maes_df = summarized_fold_maes_bh[lag]

    label = preds_df[preds_df['dvg'] == dvg_id]['label'].iloc[0]
    color = granger_color(label)

    restr_preds = preds_df[['dpi', 'restr_preds']].drop_duplicates()
    restr_maes = maes_df[['dpi', 'restr_mae']].drop_duplicates()

    fig, (ax_pred, ax_mae) = plt.subplots(1, 2, figsize=figsize)

    # Forecast plot
    sns.lineplot(x='dpi', y='actual', color='black', data=preds_df,
                ax=ax_pred, marker='s', alpha=0.5, label='Actual')
    sns.lineplot(x='dpi', y='full_preds', color=color,
                data=preds_df[preds_df['dvg'] == dvg_id],
                ax=ax_pred, marker='o', alpha=0.8, label=f'Full model ({dvg_id})')
    sns.lineplot(x='dpi', y='restr_preds', color='tab:orange',
                data=restr_preds, ax=ax_pred, marker='D', label='Restricted Model')

    ax_pred.set_title(f'Forecast — {dvg_id} ({label})')
    ax_pred.set_xlabel('DPI')
    ax_pred.set_ylabel('log10(PFU/mL+1)')
    ax_pred.set_ylim(-1, 10)
    ax_pred.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=1)

    # MAE plot
    sns.lineplot(x='dpi', y='full_mae', color=color,
                data=maes_df[maes_df['dvg'] == dvg_id],
                ax=ax_mae, marker='o', alpha=0.8, label=f'Full model ({dvg_id})')
    sns.lineplot(x='dpi', y='restr_mae', color='tab:orange',
                data=restr_maes, ax=ax_mae, marker='D', label='Restricted Model')

    ax_mae.set_title(f'MAE — {dvg_id} ({label})')
    ax_mae.set_xlabel('DPI')
    ax_mae.set_ylabel('MAE')
    ax_mae.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=1)

    # Fold lines on both
    fold_max_list = list(fold_max_dpi.values())
    for ax in (ax_pred, ax_mae):
        ax.vlines(fold_max_list, *ax.get_ylim(), colors='gray', linestyles='dashed', alpha=0.5)

    fig.tight_layout()
    plt.show()
    
    from sklearn.preprocessing import MinMaxScaler

def plot_ranked_distributions(df, num_cols, color_col='label_priority',
                              color_map=None, figsize_per_col=2.5, height=10,
                              display_names = {
      'label_priority': 'Label Priority',
      'ssr_chi2test_pval': 'SSR Chi2 p-val',
      'full_mae_median': 'MAE (median)',
      'full_mae_mad': 'MAE (MAD)',
      'ssr': 'SSR'
    }):
    if color_map is None:
        color_map = {0: 'tab:purple', 1: 'tab:blue', 2: 'tab:red'}

    scaled = pd.DataFrame(
        MinMaxScaler().fit_transform(df[num_cols]),
        columns=num_cols
    )
    scaled['row_index'] = range(len(scaled))

    melted = scaled.melt(id_vars='row_index', var_name='column', value_name='value')

    row_index2color = {
        idx: color_map.get(val, 'lightgrey')
        for idx, val in zip(df.index, df[color_col])
    }

    fig, axes = plt.subplots(1, len(num_cols),
                              figsize=(figsize_per_col * len(num_cols), height),
                              sharey=True)

    for ax, col in zip(axes, num_cols):
        col_data = melted[melted['column'] == col]
        colors = col_data['row_index'].map(row_index2color)
        colors.fillna('tab:gray', inplace=True)

        ax.scatter(col_data['value'], col_data['row_index'], s=5, alpha=0.9, color=colors)
        ax.scatter(col_data['value'], col_data['row_index'], s=5, color=None, edgecolor=colors, linewidth=0.5, alpha=1)
        for i in range(len(col_data) - 1):
            row_slice = col_data.iloc[i:i+2]
            ax.fill_betweenx(row_slice['row_index'], 0, row_slice['value'],
                            alpha=0.2, color=colors.iloc[i])

        ax.set_title(col, rotation=45, ha='left')
        ax.set_xlim(0, 1)

        orig_min = df[col].min()
        orig_max = df[col].max()
        orig_mid = (orig_min + orig_max) / 2
        ax.set_xticks([0, 0.5, 1])
        ax.set_xticklabels([f'{orig_min:.2g}', f'{orig_mid:.2g}', f'{orig_max:.2g}'])
        ax.set_xlabel(display_names.get(col, col))

    axes[0].set_ylabel('row index (ranked)')
    axes[0].invert_yaxis()
    plt.tight_layout()
    plt.show()

In [ ]:
ranked_dvgs = avg_summarized_maes_corrected[1][['label', 'ssr_chi2test_pval',
                                                'restr_mae_median', 'restr_mae_mad',
                                                'full_mae_median', 'full_mae_mad',
                                                ]].copy()
ranked_dvgs

In [ ]:
ref_ols_performance_df = pd.read_csv(f'{output_prefix}/ref_ols_performance_dct.csv')
ref_ols_performance_df

ref_ols_performance_dct = ref_ols_performance_df.set_index('lag')['SSR'].to_dict()
ref_ols_performance_dct

In [ ]:
ranked_dvgs = {}
dvg2ssr = {}

label_priority = ['bi-directional', 'causing', 'caused', 'non-related', 'shuffled', 'bootstrapped']
for lag in range(1, 2):
  dvg2ssr[lag] = gc_prediction_summary_corrected[lag].set_index('key')['ssr'].to_dict()
  ranked_dvgs[lag] = avg_summarized_maes_corrected[lag][['label', 'ssr_chi2test_pval',
                                                'restr_mae_median', 'restr_mae_mad',
                                                'full_mae_median', 'full_mae_mad'
                                                ]].copy()
  ranked_dvgs[lag]['ssr'] = ranked_dvgs[lag].index.map(dvg2ssr[lag])
  ranked_dvgs[lag]['ssr_diff'] = ranked_dvgs[lag]['ssr'] - ref_ols_performance_dct[lag]
  ranked_dvgs[lag]['label_priority'] = ranked_dvgs[lag]['label'].apply(lambda x: label_priority.index(x) if x in label_priority else len(label_priority))
  ranked_dvgs[lag] = ranked_dvgs[lag].sort_values(by=['full_mae_median', 'ssr_chi2test_pval', 'ssr',
                                                      'full_mae_mad',
                                                      'label_priority'], 
                                                  ascending=[True, True, True, True, True])
  ranked_dvgs[lag] = ranked_dvgs[lag].reset_index()

ranked_dvgs[1]

# drop if ssr is nan
ranked_dvgs[1] = ranked_dvgs[1][~ranked_dvgs[1]['ssr'].isna()]
ranked_dvgs[1]



In [ ]:
df = ranked_dvgs[1].copy()

num_cols = ['label_priority','ssr_chi2test_pval',
            'full_mae_median', 'full_mae_mad', 'ssr_diff']

fig, axes = plt.subplots(1, len(num_cols), figsize=(4 * len(num_cols), 6), sharey=False)

for ax, col in zip(axes, num_cols):
    sns.violinplot(data=df, y='label', x=col, ax=ax, cut=0, inner='quart', scale='width')
    ax.set_title(col, fontsize=10)
    ax.set_ylabel('')
    ax.set_xlabel('')

axes[0].set_ylabel('label')
fig.suptitle('Distributions by label', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
plt.rcParams.update({'font.size': 14})
plot_ranked_distributions(ranked_dvgs[1].sort_values(by=['full_mae_median', 'ssr_chi2test_pval', 'ssr',
                                                      'full_mae_mad',
                                                      'label_priority'], 
                                                  ascending=[True, True, True, True, True]),
                          num_cols=['ssr_chi2test_pval', 'full_mae_median', 'full_mae_mad', 'ssr_diff'], color_col='label_priority')


In [ ]:
plot_ranked_distributions(ranked_dvgs[1].sort_values(by=['ssr_chi2test_pval', 'full_mae_median','ssr',
                                                      'full_mae_mad',
                                                      'label_priority'], 
                                                  ascending=[True, True, True, True, True]),
                          num_cols=['ssr_chi2test_pval', 'full_mae_median', 'full_mae_mad', 'ssr_diff'], color_col='label_priority')

In [ ]:
# Rank used in the paper

ranked_dvgs[1]['norm_full_mae_median'] = ranked_dvgs[1]['full_mae_median'] / ranked_dvgs[1]['full_mae_median'].max()
ranked_dvgs[1]['norm_ssr'] = ranked_dvgs[1]['ssr'] / ranked_dvgs[1]['ssr'].max()
ranked_dvgs[1]['sum_norm_mae_ssr'] = ranked_dvgs[1]['norm_full_mae_median'] + ranked_dvgs[1]['norm_ssr']
ranked_dvgs[1]['rank_ssr_mae'] = ranked_dvgs[1]['sum_norm_mae_ssr'].rank(method='average', ascending=True)
plot_ranked_distributions(ranked_dvgs[1].sort_values(by=['rank_ssr_mae', 'ssr_chi2test_pval', 'full_mae_median', 'ssr',
                                                      'full_mae_mad',
                                                      'label_priority'],
                                                  ascending=[True, True, True, True, True, True]),
                          num_cols=['ssr_chi2test_pval', 'full_mae_median', 'full_mae_mad', 'ssr_diff'], color_col='label_priority')

In [ ]:
# alternative rank that is better for non-parametric 
df = ranked_dvgs[1].copy()
df['rank_score'] = (df['norm_ssr'].rank() + df['norm_full_mae_median'].rank()) / 2
df = df.sort_values('rank_score')
plot_ranked_distributions(df,
                          num_cols=['ssr_chi2test_pval', 'full_mae_median', 'full_mae_mad', 'ssr_diff'], color_col='label_priority')

# export files

In [ ]:
ranked_df = df.sort_values(by=['rank_ssr_mae',
                                          'ssr_chi2test_pval', 'full_mae_median', 'full_mae_mad', 'ssr',
                                            'label_priority']).reset_index(drop=True)
ranked_df

In [ ]:
ranked_df[ranked_df.dvg.isin(['PB2_269_2202', 'PB2_217_2204', 'PB2_129_2176'])]

In [ ]:
ranked_df[ranked_df.dvg == 'PB1_146_2056']

In [ ]:
ranked_df.head(10)

In [ ]:
ranked_df.rank_ssr_mae.max()

In [ ]:
ranked_df['mae_median_diff'] = ranked_df['full_mae_median'] - ranked_df['restr_mae_median']

In [ ]:
selected_dvgs = ranked_df[ranked_df['label'].isin(['causing', 'bi-directional'])][['dvg', 'label', 'rank_ssr_mae', 'full_mae_median', 'full_mae_mad', 'ssr', 'ssr_chi2test_pval', 'ssr_diff', 'mae_median_diff']].reset_index(drop=True)
selected_dvgs[['dvg', 'label', 'full_mae_median', 'full_mae_mad', 'mae_median_diff', 'ssr', 'ssr_diff', 'rank_ssr_mae']].to_csv(f'{output_prefix}/selected_dvgs_lag1.csv', index=False)
ranked_df[['dvg', 'label', 'full_mae_median', 'full_mae_mad', 'mae_median_diff', 'ssr', 'ssr_diff', 'rank_ssr_mae']].to_csv(f'{output_prefix}/ranked_dvgs_lag1.csv', index=False)


In [ ]:
selected_dvgs

In [ ]:
selected_dvgs[selected_dvgs.dvg.isin(['PB2_269_2202', 'PB2_217_2204', 'PB2_129_2176'])]

In [ ]:
for dvg in selected_dvgs['dvg'].head(10):
  plot_dvg_forecast_and_mae(dvg_id=dvg, lag=1, figsize=(12,6))